In [ ]:
EXPERIMENT_ID = '2026-07-09_18-42-14_full'
WEIGHTS_ID = -1

BATCH_SIZE = 10_000
CLIP_VAL = 10

In [ ]:
import numpy as np
import anndata as ad
import numpy as np
import json
from types import SimpleNamespace
import os
import tensorflow as tf
from tqdm import tqdm

from constants import DATA_ROOT, RESULTS_ROOT
from model import RNABagModel
from data_operations import simple_scipy_norm_x

In [ ]:
with open(RESULTS_ROOT / 'training' / EXPERIMENT_ID / 'config.json', 'r', encoding='utf-8') as fr:
    CONFIG = json.load(fr)
CF = SimpleNamespace(**CONFIG)

m = RNABagModel(n_vars=CF.n_genes, n_layers=CF.n_layers, emb_dim=CF.emb_dim)

In [ ]:
weights_list = os.listdir(RESULTS_ROOT / 'training' / EXPERIMENT_ID / 'weights')
weights_list = [w_name for w_name in weights_list if w_name.endswith('.weights.h5')]
weights_list.sort()
weights_to_load = weights_list[WEIGHTS_ID]
print(weights_to_load)
m.model.load_weights(RESULTS_ROOT / 'training' / EXPERIMENT_ID / 'weights' / weights_to_load)

In [ ]:
embedder = tf.keras.Model(
    inputs=m.model.input,
    outputs=m.model.layers[-2].output
)

In [ ]:
SRC_DIR = DATA_ROOT / 'shuffled_shards'
shards = list(SRC_DIR.glob("*.h5ad"))
shards.sort()

OUTPUT_DIR = DATA_ROOT / 'shuffled_shards_emb'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@tf.function
def emb_prediction(nn_model, x_input):
    return nn_model(x_input, training=False)

for shard_path in tqdm(shards[:50]):
    data_shard = ad.read_h5ad(shard_path)
    xxx = data_shard.X[:, :CF.n_genes]

    shard_embs = np.zeros((xxx.shape[0], CF.emb_dim), dtype='float16')

    for i in range(0, xxx.shape[0], BATCH_SIZE):
        x = xxx[i:i+BATCH_SIZE].copy()
        x = simple_scipy_norm_x(x)
        this_emb = emb_prediction(embedder, x).numpy()

        shard_embs[i:i+BATCH_SIZE] = this_emb

    shard_embs = np.clip(shard_embs, -CLIP_VAL, CLIP_VAL)
    np.save(OUTPUT_DIR / f'{shard_path.stem}.npy', shard_embs)